In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import pickle

# 1. Load 2+ years of data
df = fetch_load_data('DE', '20220101', '20241231')

# 2. Scale
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[['load_MW']])

# 3. Create sliding windows (seq_len=168 = one week of hourly data)
SEQ_LEN = 168
HORIZON = 24   # forecast 24 hours ahead

def make_sequences(data, seq_len, horizon):
    X, y = [], []
    for i in range(len(data) - seq_len - horizon):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len:i+seq_len+horizon])
    return np.array(X), np.array(y)

X, y = make_sequences(scaled, SEQ_LEN, HORIZON)

# 4. Split: 70/15/15
n = len(X)
X_train, y_train = X[:int(.7*n)], y[:int(.7*n)]
X_val,   y_val   = X[int(.7*n):int(.85*n)], y[int(.7*n):int(.85*n)]
X_test,  y_test  = X[int(.85*n):], y[int(.85*n):]

# 5. Build LSTM
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.2),
    Dense(HORIZON)
])
model.compile(optimizer='adam', loss='mse')

es = EarlyStopping(patience=5, restore_best_weights=True)
model.fit(X_train, y_train, validation_data=(X_val, y_val),
          epochs=50, batch_size=64, callbacks=[es])

# 6. Evaluate
from sklearn.metrics import mean_absolute_error
import numpy as np

preds = model.predict(X_test)
preds_inv = scaler.inverse_transform(preds)
y_inv     = scaler.inverse_transform(y_test)

mae  = mean_absolute_error(y_inv.flatten(), preds_inv.flatten())
rmse = np.sqrt(np.mean((y_inv - preds_inv)**2))
mape = np.mean(np.abs((y_inv - preds_inv) / y_inv)) * 100
print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}, MAPE: {mape:.2f}%")

# 7. Save
model.save('models/lstm_model.h5')
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)